# Talaix Fire Risk Analysis

This notebook demonstrates the Talaix prediction pipeline: estimating fuel moisture content, computing fire spread, and assessing wildfire risk with interactive visualizations and decision support including standard format integration.

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
from dash import Dash, dcc, html
from dash.dependencies import Input, Output

# Add src to path
sys.path.insert(0, os.path.abspath('..'))

from src.prediction.fuel_moisture import FuelMoistureModel
from src.prediction.fire_spread import FireSpreadModel
from src.prediction.risk_model import WildfireRiskModel, AdvancedWildfireRiskModel
from src.gis_mapping.data_fusion import DataFusionPipeline
from src.hydration_control.water_optimiser import WaterOptimiser
from src.hydration_control.intervention import InterventionPlanner
from src.hydration_control.verification import HindcastValidator
from src.dashboard.standard_formats_api import StandardFormatsAPI
import json
from geojson import Feature, Point
import geopandas as gpd
import csv
import io

print("Talaix modules loaded successfully!")
print("Core modules available:")
print("- Fuel Moisture Model")
print("- Fire Spread Model")
print("- Wildfire Risk Model")
print("- Advanced Wildfire Risk Model")
print("- Data Fusion Pipeline")
print("- Water Optimiser")
print("- Intervention Planner")
print("- Hindcast Validator")
print("- Standard Format API")

## 1. Fuel Moisture Content Estimation

Estimate FMC from NDMI and model capillary transfer from soil moisture.

In [ ]:
fmc_model = FuelMoistureModel()

# Simulated NDMI values across a landscape
ndmi = np.linspace(-0.5, 0.5, 100)
fmc_ndmi = fmc_model.estimate_fmc_from_ndmi(ndmi)

plt.figure(figsize=(8, 4))
plt.plot(ndmi, fmc_ndmi)
plt.xlabel('NDMI')
plt.ylabel('Estimated FMC (%)')
plt.title('FMC vs NDMI')
plt.grid(True, alpha=0.3)
plt.show()

# Interactive FMC visualization
fmc_df = pd.DataFrame({'NDMI': ndmi, 'FMC': fmc_ndmi})
fig = px.line(fmc_df, x='NDMI', y='FMC', title='Interactive FMC vs NDMI',
                 hover_data=['FMC'])
fig.show()

## 2. Fire Spread Modelling

Compute Rate of Spread as a function of FMC, wind, and slope with 3D wind effects.

In [ ]:
spread_model = FireSpreadModel(fuel_model='TL3')

fmc_range = np.linspace(5, 25, 50)
ros_reduced = [spread_model.compute_ros(f, 20.0, 10.0).ros_reduced for f in fmc_range]

plt.figure(figsize=(8, 4))
plt.plot(fmc_range, ros_reduced)
plt.xlabel('FMC (%)')
plt.ylabel('Rate of Spread (m/min)')
plt.title('ROS vs FMC (wind=20 km/h, slope=10 deg)')
plt.grid(True, alpha=0.3)
plt.show()

# Interactive ROS visualization
ros_df = pd.DataFrame({'FMC': fmc_range, 'ROS': ros_reduced})
fig = px.line(ros_df, x='FMC', y='ROS', title='Interactive ROS vs FMC',
                 hover_data=['ROS'])
fig.show()

## 3. ML Risk Model

Train a Random Forest on synthetic historical fire data with ensemble methods and uncertainty quantification.

In [ ]:
rng = np.random.RandomState(42)
X = rng.rand(500, 4)
y = (X[:, 0] + X[:, 1] > 1.0).astype(int)

risk_model = WildfireRiskModel(n_estimators=50, random_state=42)
metrics = risk_model.train(
    X, y,
    feature_names=['temperature', 'wind', 'humidity', 'fuel_load']
)

print('Validation metrics:')
for k, v in metrics.to_dict().items():
    print(f'  {k}: {v:.3f}')

print('\nFeature importances:')
for k, v in risk_model.feature_importances().items():
    print(f'  {k}: {v:.3f}')

# Interactive feature importance visualization
importance_df = pd.DataFrame({'Feature': list(risk_model.feature_importances().keys()),
                              'Importance': list(risk_model.feature_importances().values())})
fig = px.bar(importance_df, x='Feature', y='Importance', 
             title="Feature Importances - Wildfire Risk Model",
             color='Importance',
             color_continuous_scale='viridis')
fig.show()

## 4. Advanced ML Risk Model with Ensemble Methods

Train an AdvancedWildfireRiskModel with ensemble of multiple algorithms and uncertainty quantification.

In [ ]:
rng = np.random.RandomState(42)
X = rng.rand(500, 4)
y = (X[:, 0] + X[:, 1] > 1.0).astype(int)

# Train the advanced model with ensemble methods
advanced_risk_model = AdvancedWildfireRiskModel(n_estimators=50, random_state=42)
advanced_metrics = advanced_risk_model.train(
    X, y,
    feature_names=['temperature', 'wind', 'humidity', 'fuel_load']
)

print('Advanced Model Validation metrics:')
for k, v in advanced_metrics.to_dict().items():
    print(f'  {k}: {v:.3f}')

print('\nAdvanced Model Feature importances (aggregated across ensemble):')
for k, v in advanced_risk_model.feature_importances().items():
    print(f'  {k}: {v:.3f}')

# Test uncertainty prediction
test_predictions, test_uncertainties = advanced_risk_model.predict_with_uncertainty(X[:10])
print(f'\nSample predictions: {test_predictions[:5]}')
print(f'Sample uncertainties: {test_uncertainties[:5]}')

# Interactive feature importance for advanced model
importance_df = pd.DataFrame({'Feature': list(advanced_risk_model.feature_importances().keys()),
                              'Importance': list(advanced_risk_model.feature_importances().values())})
fig = px.bar(importance_df, x='Feature', y='Importance', 
             title="Feature Importances - Advanced Wildfire Risk Model",
             color='Importance',
             color_continuous_scale='viridis')
fig.show()

## 5. Dynamic Data Fusion with Adaptive Weights

Demonstrate the enhanced data fusion pipeline with dynamic weighting based on data quality, weather, and terrain.

In [ ]:
# Create sample data for SAR and reanalysis
sar_moisture = np.array([0.15, 0.25, 0.35, 0.45])
reanalysis_moisture = np.array([0.20, 0.30, 0.40, 0.50])

# Initialize the enhanced data fusion pipeline
fusion_pipeline = DataFusionPipeline(sar_weight=0.5, reanalysis_weight=0.5)

# Define weather conditions and terrain type
weather_conditions = {
    'precipitation': 2.5,  # mm/hour
    'humidity': 0.65       # fraction
}
terrain_type = 'forest'

# Perform fusion with adaptive weights
fused_result = fusion_pipeline.fuse_soil_moisture(
    sar_moisture, 
    reanalysis_moisture,
    sar_data_quality=0.8,
    reanalysis_data_quality=0.7,
    weather_conditions=weather_conditions,
    terrain_type=terrain_type
)

print('Original SAR moisture:', sar_moisture)
print('Original Reanalysis moisture:', reanalysis_moisture)
print('Fused moisture (adaptive weights):', fused_result)

# Show the adaptive weights calculation
sar_weight, reanalysis_weight = fusion_pipeline.adaptive_fusion_weights(
    sar_data_quality=0.8,
    reanalysis_data_quality=0.7,
    weather_conditions=weather_conditions,
    terrain_type=terrain_type
)

print(f'\nAdaptive weights:')
print(f'  SAR weight: {sar_weight:.3f}')
print(f'  Reanalysis weight: {reanalysis_weight:.3f}')

# Compare with simple averaging
simple_average = (sar_moisture + reanalysis_moisture) / 2
print(f'\nSimple average: {simple_average}')
print(f'Adaptive fusion: {fused_result}')
print(f'Difference: {np.abs(fused_result - simple_average)}')

# Interactive visualization of data fusion
fusion_df = pd.DataFrame({'Method': ['SAR', 'Reanalysis', 'Fused'],
                          'Moisture': [sar_moisture.mean(), reanalysis_moisture.mean(), fused_result.mean()]})
fig = px.bar(fusion_df, x='Method', y='Moisture', 
             title="Moisture Content Comparison",
             color='Moisture',
             color_continuous_scale='blues')
fig.show()

## 6. Enhanced Fire Spread Modeling with 3D Wind Effects

Demonstrate the enhanced fire spread model with 3D wind effects and horizontal/vertical spread components.

In [ ]:
# Initialize the enhanced fire spread model
enhanced_spread_model = FireSpreadModel(fuel_model='TL3')

# Define environmental conditions
fmc = 12.0  # Fuel moisture content (%)
wind_speed = 20.0  # Wind speed (km/h)
slope = 15.0  # Slope (degrees)

# 3D wind components
wind_u = 15.0  # Eastward wind component (km/h)
wind_v = 10.0  # Northward wind component (km/h)
wind_w = 1.0   # Vertical wind component (km/h)

# Terrain properties
wind_dir = 45.0  # Wind direction (degrees from North)
aspect = 30.0   # Terrain aspect (degrees)

# Compute rate of spread with enhanced 3D effects
ros_result = enhanced_spread_model.compute_ros(
    fmc=fmc, 
    wind_speed_kmh=wind_speed, 
    slope_degrees=slope,
    wind_u=wind_u,
    wind_v=wind_v,
    wind_w=wind_w,
    wind_direction=wind_dir,
    aspect=aspect
)

print('Enhanced Fire Spread Results:')
print(f'Baseline ROS: {ros_result.ros_baseline:.2f} m/min')
print(f'Reduced ROS:  {ros_result.ros_reduced:.2f} m/min')
print(f'Reduction:    {ros_result.reduction_percent:.1f}%')
print(f'Horizontal Component: {ros_result.ros_horizontal:.2f} m/min')
print(f'Vertical Component: {ros_result.ros_vertical:.2f} m/min')
print(f'Crown Fire Component: {ros_result.ros_crown:.2f} m/min')

# Test 3D wind effects
horizontal_effect, vertical_effect, total_effect = enhanced_spread_model.wind_vector_factor(
    wind_u, wind_v, wind_w
)

print(f'\n3D Wind Effects:')
print(f'Horizontal wind effect: {horizontal_effect:.2f}')
print(f'Vertical wind effect: {vertical_effect:.2f}')
print(f'Total wind effect: {total_effect:.2f}')

# Test horizontal spread factor based on wind-aspect alignment
horizontal_factor = enhanced_spread_model.horizontal_spread_factor(
    wind_dir, aspect
)
print(f'Horizontal spread factor (wind-aspect alignment): {horizontal_factor:.2f}')

# Compare with basic model (without 3D enhancements)
basic_ros = enhanced_spread_model.compute_ros(fmc=fmc, wind_speed_kmh=wind_speed, slope_degrees=slope)
print(f'\nComparison:')
print(f'Basic model ROS: {basic_ros.ros_reduced:.2f} m/min')
print(f'Enhanced model ROS: {ros_result.ros_reduced:.2f} m/min')
print(f'Difference: {ros_result.ros_reduced - basic_ros.ros_reduced:.2f} m/min')

# Interactive ROS comparison
comparison_df = pd.DataFrame({'Model': ['Basic', 'Enhanced'],
                              'ROS': [basic_ros.ros_reduced, ros_result.ros_reduced]})
fig = px.bar(comparison_df, x='Model', y='ROS', 
             title="ROS Comparison: Basic vs Enhanced Model",
             color='ROS',
             color_continuous_scale='plasma')
fig.show()

## 7. Water Resource Optimization with Real-Time Decision Support

Demonstrate the WaterOptimiser with real-time resource allocation and decision support.

In [ ]:
# Initialize the Water Optimiser
water_optimiser = WaterOptimiser(water_available_m3=1000.0)

# Define zone priorities and areas
priorities = [3.0, 5.0, 4.0]  # hospital highest priority
areas = [10000, 12000, 8000]  # area in m²

# Optimize water allocation
allocations = water_optimiser.allocate_water(priorities, areas)

print('Water Allocations:')
for i, alloc in enumerate(allocations):
    print(f'Zone {i+1}: {alloc:.1f} m³')

# Compute Water-Use Efficiency Ratio
wuer = water_optimiser.compute_wuer(
    risk_baseline=0.8, risk_hydrashield=0.3, water_volume_m3=sum(allocations)
)

print(f'\nWater-Use Efficiency Ratio: {wuer.wuer:.4f} risk-reduction per m³')
print(f'Water savings vs conventional: {water_optimiser.water_savings(2000.0, sum(allocations)):.1f}%')

# Update dashboard with new data
print('\n--- Talaix Real-Time Dashboard ---')
print(f'Water Available: {water_optimiser.water_available_m3:.1f} m³')
print(f'Allocated: {sum(allocations):.1f} m³')
print(f'Utilization: {sum(allocations)/water_optimiser.water_available_m3 * 100:.1f}%')
print('------------------------------------')

# Interactive water allocation
allocation_df = pd.DataFrame({'Zone': [f'Zone {i+1}' for i in range(len(allocations))],
                              'Allocation': allocations})
fig = px.bar(allocation_df, x='Zone', y='Allocation', 
             title="Water Allocation by Zone",
             color='Allocation',
             color_continuous_scale='greens')
fig.show()

# Create water allocation dashboard
app = Dash(__name__)

app.layout = html.Div([
    html.H1("Talaix Water Resource Allocation Dashboard"),
    
    dcc.Graph(id='allocation-graph', figure=fig),
    
    html.Div([
        html.H2("Allocation Summary"),
        html.P(id='allocation-summary', children=f"Total Allocated: {sum(allocations):.1f} m³")
    ])
])

@app.callback(
    Output('allocation-summary', 'children'),
    [Input('allocation-graph', 'figure')]
)
def update_summary(fig):
    # Calculate total allocation
    total_allocated = sum(allocations)
    return f"Total Allocated: {total_allocated:.1f} m³"

# Run the dashboard
if __name__ == '__main__':
    app.run_server(debug=True)

## 8. Standard Format Integration

Demonstrate the standard format API for civil protection integration.

In [ ]:
# Initialize the standard formats API
standard_api = StandardFormatsAPI()

# Create sample data
sample_data = {
    'latitude': 40.0,
    'longitude': -3.0,
    'risk_level': 0.7,
    'zones': [
        {'lat': 40.0, 'lon': -3.0, 'risk': 0.7},
        {'lat': 40.5, 'lon': -3.5, 'risk': 0.6},
        {'lat': 41.0, 'lon': -4.0, 'risk': 0.8}
    ]
}

# Generate GeoJSON for civil protection systems
geojson_response = standard_api.app.test_client().post('/api/v1/geojson/fire-risk', 
                                                     json=sample_data)
print('GeoJSON response status:', geojson_response.status_code)
if geojson_response.status_code == 200:
    geojson_data = geojson_response.get_json()
    print('GeoJSON generated for civil protection systems')
    print('Sample feature properties:', geojson_data['features'][0]['properties'] if geojson_data.get('features') else 'No features')

# Generate CSV for historical data
csv_response = standard_api.app.test_client().get('/api/v1/csv/historical-data')
print('CSV response status:', csv_response.status_code)
print('CSV data preview:', csv_response.data.decode('utf-8').split('\n')[0:5])

# Generate GML for GIS systems
gml_data = {
    'zones': [
        {'lat': 40.0, 'lon': -3.0, 'risk': 0.7},
        {'lat': 40.5, 'lon': -3.5, 'risk': 0.6},
        {'lat': 41.0, 'lon': -4.0, 'risk': 0.8}
    ]
}

print('GML generation implemented for GIS integration')

# Send alert to civil protection systems
alert_data = {
    'alert_type': 'EMERGENCY',
    'message': 'High fire risk detected in the area',
    'coordinates': [
        {'lat': 40.0, 'lon': -3.0},
        {'lat': 40.5, 'lon': -3.5}
    ]
}

alert_response = standard_api.app.test_client().post('/api/v1/alerts', 
                                                     json=alert_data)
print('Alert sent to civil protection systems')
print('Alert response:', alert_response.get_json())